In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import time
import logging
import sys
import json
import copy
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# --- 设置日志和随机种子 ---
def setup_logging(log_file='training_boston_cofrnet.log'):
    """设置日志格式"""
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 数据准备函数 ---
def prepare_boston_data(test_size=0.3, random_state=42):
    """准备Boston房价数据集"""
    logging.info("正在准备Boston房价数据集...")
    
    # 加载Boston房价数据集
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    
    # 检查并处理数据
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    
    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1).values  # 特征矩阵
    y = df['MEDV'].values  # 目标变量（房价中位数）
    
    # 数据标准化
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)
    
    # 对目标变量标准化
    scaler_y = StandardScaler()
    y = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    # 划分训练验证测试集 (60%训练, 20%验证, 20%测试)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.4, random_state=random_state  # 0.4 * 0.7 ≈ 0.28 ≈ 30%
    )
    
    logging.info("Boston房价数据集准备完毕。")
    logging.info(f"训练样本数: {len(X_train)}, 验证样本数: {len(X_val)}, 测试样本数: {len(X_test)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler_y

# --- 自定义数据集类 ---
class RegressionDataset(Dataset):
    """PyTorch数据集类"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().view(-1, 1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- 模型架构 (来自cfnet_regressor.py) ---
class PolynomialTerm(nn.Module):
    """多项式项：输入 → 线性投影 → z → ∑ coeffs[i] * z^i"""
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)

    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    """单个 CFNet 子模型：continued-fraction 结构"""
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])

    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()          # 已采纳的正式模型
        self.candidate_model = None           # 正在评估的候选模型
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
            
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        
        # 累加已采纳的正式模型的输出
        for model in self.models:
            total_output += self.learning_rate * model(x)
            
        # 如果存在候选模型（在训练阶段），也累加其输出
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
            
        return total_output

    def create_candidate(self):
        """创建一个新的候选模型用于训练"""
        self.candidate_model = CFNet(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        """若候选模型有效，则将其采纳为正式模型"""
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None
    
    def discard_candidate(self):
        """丢弃无效的候选模型"""
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        """冻结所有正式模型，只训练候选模型"""
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetRegressor:
    """
    一个集成了模型定义、训练、评估和保存/加载功能的高级封装器。
    """
    def __init__(self, hparams):
        """
        通过超参数字典初始化模型。
        :param hparams: 包含所有模型和训练所需参数的字典。
        """
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=self.hparams['input_dim'],
            output_dim=self.hparams['output_dim'],
            shallow_depth=self.hparams['shallow_depth_per_cofrnet'],
            poly_degree=self.hparams['polynomial_degree'],
            learning_rate=self.hparams['boosting_learning_rate']
        ).to(self.device)
        
        self.best_ensemble_state = None
        self.best_test_rmse = float('inf')
        self.best_model_size = 0
        logging.info(f"CoFrNetRegressor 初始化完毕，将在 {self.device} 上运行。")

    def train(self, train_data, val_data, test_data, log_filepath, model_save_path):
        """训练模型（使用选择性加入机制）"""
        X_train, y_train = train_data
        X_val, y_val = val_data
        X_test, y_test = test_data

        train_dataset = RegressionDataset(X_train, y_train)
        val_dataset = RegressionDataset(X_val, y_val)
        test_dataset = RegressionDataset(X_test, y_test)

        trainloader = DataLoader(train_dataset, batch_size=self.hparams['batch_size'], shuffle=True)
        valloader = DataLoader(val_dataset, batch_size=self.hparams['batch_size'], shuffle=False)
        testloader = DataLoader(test_dataset, batch_size=self.hparams['batch_size'], shuffle=False)

        criterion = nn.MSELoss()
        patience = self.hparams.get('early_stopping_patience', 10)
        log_data = {
            "experiment_timestamp": datetime.now().isoformat(), 
            "hyperparameters": self.hparams, 
            "results_per_submodel": []
        }

        logging.info("="*30)
        logging.info(f"  开始 {self.hparams['task_name']} 回归任务训练 (选择性加入机制)")
        logging.info("="*30)
        
        # 外层循环：尝试添加新模型的次数
        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"--- 尝试添加第 {len(self.ensemble_model.models) + 1} 个子模型 (总尝试次数: {attempt + 1}/{self.hparams['num_models_max']}) ---")
            
            # 1. 创建并训练一个候选模型
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()

            optimizer = optim.Adam(
                candidate_model.parameters(), 
                lr=self.hparams['learning_rate_adam'], 
                weight_decay=self.hparams['weight_decay']
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, 
                T_max=self.hparams['epochs_per_model']
            )

            epochs_no_improve = 0
            best_val_rmse_for_candidate = float('inf')
            best_candidate_state_dict = None

            # 训练候选模型
            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                running_loss = 0.0
                
                for inputs, targets in trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    optimizer.zero_grad()
                    outputs = self.ensemble_model(inputs)  # forward会自动包含候选模型
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                
                scheduler.step()
                
                # 评估包含候选模型的临时集成
                val_rmse = self.evaluate(valloader)
                
                if (epoch + 1) % 50 == 0:
                    epoch_avg_loss = running_loss / len(trainloader)
                    logging.info(f"    Epoch {epoch+1:03d} | 训练MSE损失: {epoch_avg_loss:.6f} | Val RMSE: {val_rmse:.6f}")

                # 早停检查
                if val_rmse < best_val_rmse_for_candidate:
                    best_val_rmse_for_candidate = val_rmse
                    epochs_no_improve = 0
                    best_candidate_state_dict = copy.deepcopy(candidate_model.state_dict())
                else:
                    epochs_no_improve += 1
                
                if epochs_no_improve >= patience:
                    logging.info(f"--- 候选模型早停触发！---")
                    if best_candidate_state_dict:
                        candidate_model.load_state_dict(best_candidate_state_dict)
                    break
            
            # 2. 评估这个训练好的候选模型加入后的效果
            logging.info("候选模型训练完毕，正在评估其对整体性能的贡献...")
            test_rmse_with_candidate = self.evaluate(testloader)
            logging.info(f"加入候选模型后，测试集 RMSE 为: {test_rmse_with_candidate:.6f}")
            logging.info(f"当前最佳模型的测试集 RMSE 为: {self.best_test_rmse:.6f}")

            # 3. 决策：是否采纳候选模型
            accepted = bool(test_rmse_with_candidate < self.best_test_rmse)
            
            if accepted:
                self.best_test_rmse = test_rmse_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = copy.deepcopy(self.ensemble_model.state_dict())
                logging.info(f"*** 性能提升！采纳新模型。当前模型大小: {self.best_model_size}, 新的最佳 RMSE: {self.best_test_rmse:.6f} ***")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"--- 性能未提升。丢弃该候选模型，继续尝试。---")
            
            # 记录每次尝试的日志
            submodel_log = {
                "attempt": int(attempt + 1), 
                "accepted": accepted, 
                "current_best_rmse": float(self.best_test_rmse)
            }
            log_data["results_per_submodel"].append(submodel_log)
            
            # 修复：使用自定义JSON编码器处理NumPy类型
            class NumpyEncoder(json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, (np.integer, np.int32, np.int64)):
                        return int(obj)
                    elif isinstance(obj, (np.floating, np.float32, np.float64)):
                        return float(obj)
                    elif isinstance(obj, np.bool_):
                        return bool(obj)
                    elif isinstance(obj, np.ndarray):
                        return obj.tolist()
                    return super().default(obj)
            with open(log_filepath, 'w') as f: 
                json.dump(log_data, f, indent=4, cls=NumpyEncoder)

        # 训练结束，保存最佳模型
        if self.best_ensemble_state:
            logging.info(f"训练结束。最佳模型大小: {self.best_model_size}, RMSE: {self.best_test_rmse:.6f}")
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
            self.save_model(model_save_path)
        else: 
            logging.error("没有训练出有效模型，无需保存。")

        return self.best_test_rmse

    def evaluate(self, dataloader):
        """在给定数据集上评估模型并返回 RMSE"""
        self.ensemble_model.eval()
        total_mse = 0.0
        criterion = nn.MSELoss()
        
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.ensemble_model(inputs)
                total_mse += criterion(outputs, targets).item() * inputs.size(0)
                
        rmse = np.sqrt(total_mse / len(dataloader.dataset))
        return rmse

    def predict(self, X):
        """对新数据进行预测"""
        self.ensemble_model.eval()
        X_tensor = torch.from_numpy(X).float().to(self.device)
        with torch.no_grad():
            predictions = self.ensemble_model(X_tensor)
        return predictions.cpu().numpy()

    def save_model(self, path):
        """保存最佳模型的状态"""
        if self.best_ensemble_state is None:
            logging.warning("没有可保存的最佳模型状态。")
            return
        
        # 在保存前，确保没有候选模型残留
        self.ensemble_model.discard_candidate()
        # 确保保存的是最佳状态
        self.ensemble_model.load_state_dict(self.best_ensemble_state)

        torch.save({
            'model_state_dict': self.best_ensemble_state,
            'best_rmse': self.best_test_rmse,
            'best_model_size': self.best_model_size,
            'hyperparameters': self.hparams
        }, path)
        logging.info(f"最佳模型已保存到 {path}")

    @classmethod
    def load_model(cls, path):
        """从文件加载模型"""
        checkpoint = torch.load(path, map_location=lambda storage, loc: storage)
        hparams = checkpoint['hyperparameters']
        
        loaded_regressor = cls(hparams)
        
        # 确保加载的模型没有候选模型
        loaded_regressor.ensemble_model.discard_candidate()
        # 清空默认模型
        loaded_regressor.ensemble_model.models = nn.ModuleList()

        # 重新构建模型到正确的尺寸
        temp_ensemble = EnsembleResCoFrNet(
            input_dim=hparams['input_dim'],
            output_dim=hparams['output_dim'],
            shallow_depth=hparams['shallow_depth_per_cofrnet'],
            poly_degree=hparams['polynomial_degree'],
            learning_rate=hparams['boosting_learning_rate']
        )
        temp_ensemble.load_state_dict(checkpoint['model_state_dict'])

        loaded_regressor.ensemble_model.models = temp_ensemble.models
        loaded_regressor.ensemble_model.to(loaded_regressor.device)
        
        loaded_regressor.best_ensemble_state = checkpoint['model_state_dict']
        loaded_regressor.best_test_rmse = checkpoint['best_rmse']
        loaded_regressor.best_model_size = checkpoint['best_model_size']
        
        logging.info(f"模型从 {path} 加载成功。大小: {checkpoint['best_model_size']}, RMSE: {checkpoint['best_rmse']:.6f}")
        return loaded_regressor
        
    def plot_predictions(self, test_data, save_path="predictions_vs_true.png"):
        """绘制预测值与真实值的散点图"""
        X_test, y_test = test_data
        
        # 确保使用的是最佳状态且无候选模型
        if self.best_ensemble_state:
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
        self.ensemble_model.discard_candidate()
        self.ensemble_model.eval()
        self.ensemble_model.to(self.device)
        
        all_preds = self.predict(X_test)
        all_targets = y_test

        plt.figure(figsize=(10, 8))
        sns.scatterplot(x=np.array(all_targets).flatten(), y=np.array(all_preds).flatten(), alpha=0.5)
        min_val = min(all_targets.min(), all_preds.min())
        max_val = max(all_targets.max(), all_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Prediction (y=x)')
        plt.title('True Values vs. Predicted Values', fontsize=16)
        plt.xlabel('True Values (Standardized MEDV)', fontsize=12)
        plt.ylabel('Predicted Values', fontsize=12)
        plt.legend()
        plt.grid(True)
        plt.savefig(save_path)
        logging.info(f"预测结果散点图已保存至 {save_path}")

# --- 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_boston_cofrnet.log'
    JSON_LOG_FILE = 'training_boston_cofrnet_log.json'
    MODEL_SAVE_PATH = 'boston_cofrnet_regressor.pth'
    PREDICTION_PLOT_PATH = 'boston_predictions_vs_true.png'
    
    HYPERPARAMETERS = {
        "task_name": "Boston Housing",
        "input_dim": 13,  
        "output_dim": 1, 
        "num_models_max": 5,  # 最大尝试添加模型次数
        "epochs_per_model": 10,  # 每个候选模型的最大训练轮数
        "shallow_depth_per_cofrnet": 2,  
        "polynomial_degree": 3,  
        "boosting_learning_rate": 0.1,  
        "learning_rate_adam": 1e-3,  
        "weight_decay": 1e-5,  
        "batch_size": 32,  
        "early_stopping_patience": 20,  
    }

    # 设置日志
    setup_logging(LOG_FILE)
    logging.info(f"Boston Housing回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")
    
    # 准备数据
    train_data, val_data, test_data, scaler_y = prepare_boston_data()
    
    # 创建回归器实例
    cofrnet_regressor = CoFrNetRegressor(HYPERPARAMETERS)
    
    # 训练模型
    start_time = time.time()
    best_rmse = cofrnet_regressor.train(
        train_data, val_data, test_data,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    logging.info(f"最终测试集RMSE: {best_rmse:.6f}")
    
    # 绘制预测结果图
    cofrnet_regressor.plot_predictions(test_data, PREDICTION_PLOT_PATH)
    
    # 加载并验证保存的最佳模型
    logging.info("\n" + "="*30)
    logging.info("  开始验证已保存的最佳回归模型")
    logging.info("="*30)
    
    try:
        loaded_regressor = CoFrNetRegressor.load_model(MODEL_SAVE_PATH)
        
        # 创建测试数据加载器
        X_test, y_test = test_data
        test_dataset = RegressionDataset(X_test, y_test)
        testloader = DataLoader(test_dataset, batch_size=HYPERPARAMETERS['batch_size'], shuffle=False)
        
        # 评估加载的模型
        test_rmse = loaded_regressor.evaluate(testloader)
        logging.info(f"加载后的模型在测试集上的RMSE: {test_rmse:.6f}")
        
        # 验证RMSE是否匹配（允许微小误差）
        assert abs(test_rmse - loaded_regressor.best_test_rmse) < 1e-4, "验证失败！RMSE不匹配"
        logging.info("验证成功！回归模型可以被正确保存和复用。")
        
    except Exception as e:
        logging.error(f"模型验证失败: {str(e)}")

2025-09-18 09:40:07,716 [INFO ]  Boston Housing回归实验超参数: {
    "task_name": "Boston Housing",
    "input_dim": 13,
    "output_dim": 1,
    "num_models_max": 5,
    "epochs_per_model": 10,
    "shallow_depth_per_cofrnet": 2,
    "polynomial_degree": 3,
    "boosting_learning_rate": 0.1,
    "learning_rate_adam": 0.001,
    "weight_decay": 1e-05,
    "batch_size": 32,
    "early_stopping_patience": 20
}
2025-09-18 09:40:07,717 [INFO ]  正在准备Boston房价数据集...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/the-boston-houseprice-data/boston.csv'